In [ ]:
# 必要パッケージ
!pip install datasets==3.6.0
!pip install haystack-ai elasticsearch-haystack

In [ ]:
## データセット用意
from datasets import load_dataset
subjqa = load_dataset("subjqa", name="electronics")
import pandas as pd
dfs = {split: dset.to_pandas() for split, dset in subjqa.flatten().items()}

In [ ]:
## Elasticsearch ダウンロード
url = "https://artifacts.elastic.co/downloads/elasticsearch/elasticsearch-7.17.10-linux-x86_64.tar.gz"
!wget -nc -q {url}
!tar -xzf elasticsearch-7.17.10-linux-x86_64.tar.gz

In [ ]:
## Elasticsearch 起動
!pkill -f org.elasticsearch.bootstrap.Elasticsearch || true
!sysctl -w vm.max_map_count=262144
!chown -R daemon:daemon /content/elasticsearch-7.17.10
!su -s /bin/bash -c 'export ES_JAVA_OPTS="-Des.cgroups.hierarchy.override=/ -XX:-UseContainerSupport"; \
/content/elasticsearch-7.17.10/bin/elasticsearch \
  -Ediscovery.type=single-node \
  -Expack.security.enabled=false \
  -Expack.ml.enabled=false \
  -Ehttp.host=127.0.0.1 \
  -Etransport.host=127.0.0.1 \
  > /tmp/es.log 2>&1 &' daemon
!sleep 25
!tail -n 120 /tmp/es.log
!curl -s http://127.0.0.1:9200

In [ ]:
### 起動確認
!curl -X GET "localhost:9200/?pretty"

In [ ]:
## Retriever の用意
### ドキュメントストアのインスタンス化
from haystack_integrations.document_stores.elasticsearch import ElasticsearchDocumentStore
custom_mapping = {
    "properties": {
        "content": {"type": "text"},
        "id": {"type": "keyword"},
        "item_id": {"type": "keyword"},
        "question_id": {"type": "keyword"},
        "split": {"type": "keyword"}
    }
}
document_store = ElasticsearchDocumentStore(hosts="http://localhost:9200",
                                            index="my_document_store",
                                            custom_mapping=custom_mapping)

### Haystack でのドキュメントストアのインデックス化
from haystack.dataclasses import Document
from haystack.document_stores.types import DuplicatePolicy
import json
def safe_str(v):
    if v is None:
        return ""
    try:
        if pd.isna(v):
            return ""
    except Exception:
        pass
    if isinstance(v, (dict, list, tuple)):
        return json.dumps(v, ensure_ascii=False)
    return str(v)
for split, df in dfs.items():
    # 重複するレビュー除去
    work = df.dropna(subset=["context"]).drop_duplicates(subset="context")
    docs = [
        Document(
            content=safe_str(row.get("context")).strip(),
            meta={
                "item_id": safe_str(row.get("title")),
                "question_id": safe_str(row.get("id")),
                "split": safe_str(split),
            },
        )
        for _, row in work.iterrows()
        if safe_str(row.get("context")).strip()
    ]
    if docs:
        written = document_store.write_documents(
            docs,
            policy=DuplicatePolicy.SKIP,
            refresh="wait_for",
        )
print(f"Loaded {document_store.count_documents()} documents")

### Elastic Search を用いたレトリーバー
from haystack_integrations.components.retrievers.elasticsearch import ElasticsearchBM25Retriever
es_retriever = ElasticsearchBM25Retriever(document_store=document_store)

In [ ]:
# Label ドキュメントのリスト作成
from haystack import Document
from haystack.document_stores.types import DuplicatePolicy
label_store = ElasticsearchDocumentStore(hosts="http://localhost:9200",
                                            index="label",
                                            custom_mapping=custom_mapping)

label_docs = []
no_ans_count = 0
for i, row in dfs["test"].iterrows():
    qid = str(row["id"])

    if not len(row["answers.text"]):
        label_docs.append(
            Document(
                content="",
                meta={
                    "type": "label",
                    "question_id": qid,
                    "item_id": row["title"],
                    "question": row["question"],
                    "answer": "",
                    "no_answer": True,
                    "row_id": int(i),
                },
            )
        )
        no_ans_count += 1
    else:
        for ans in row["answers.text"]:
            label_docs.append(
                Document(
                    content=ans,
                    meta={
                        "type": "label",
                        "question_id": qid,
                        "item_id": row["title"],
                        "question": row["question"],
                        "answer": ans,
                        "no_answer": False,
                        "row_id": int(i),
                    },
                )
            )

print(f"loaded data: {len(label_docs)}, no_ans data: {no_ans_count}")
label_store.write_documents(label_docs, policy=DuplicatePolicy.SKIP)

In [ ]:
# ラベル確認
docs = label_store.filter_documents()
print(docs[0])

In [ ]:
# item_id で集約
from collections import defaultdict
from haystack import component

@component
class LabelAggregatorByItem:
    def __init__(self, label_document_store):
        self.store = label_document_store

    @component.output_types(relevant_item_ids=list[str], answers_by_item=dict)
    def run(self, question_id: str):
        qid = str(question_id).strip()

        docs = self.store.filter_documents(
            filters={
                "operator": "AND",
                "conditions": [
                    {"field": "meta.question_id", "operator": "==", "value": qid},
                    {"field": "meta.no_answer",   "operator": "==", "value": False},
                ],
            }
        )

        agg = defaultdict(set)
        for d in docs:
            item_id = str(d.meta.get("item_id", "")).strip()
            answer  = str(d.meta.get("answer",  "")).strip()
            if item_id and answer:
                agg[item_id].add(answer)

        answers_by_item   = {k: sorted(v) for k, v in agg.items()}
        relevant_item_ids = sorted(answers_by_item.keys())
        return {"relevant_item_ids": relevant_item_ids, "answers_by_item": answers_by_item}

In [ ]:
from elasticsearch._sync.client import Elasticsearch
# 評価値計算
@component
class ItemRecallEvaluator:
    """
    Retriever 出力 documents の item_id と
    label 側 relevant_item_ids を比較して Recall@k を返す
    """
    @component.output_types(recall=float, hits=int, total_relevant=int)
    def run(self, documents: list, relevant_item_ids: list[str]):
        retrieved_item_ids = []
        for d in documents:
            item_id = d.meta.get("item_id")
            if item_id is not None:
                retrieved_item_ids.append(str(item_id))

        retrieved_set = set(retrieved_item_ids)
        relevant_set = set(relevant_item_ids)

        if not relevant_set:
            return {"recall": 0.0, "hits": 0, "total_relevant": 0}

        hits = len(retrieved_set & relevant_set)
        recall = hits / len(relevant_set)
        return {"recall": recall, "hits": hits, "total_relevant": len(relevant_set)}

# Pipeline 構築
from haystack import Pipeline
retriever = ElasticsearchBM25Retriever(document_store=document_store)

pipe = Pipeline()
pipe.add_component("Retriever", retriever)
pipe.add_component("LabelAgg", LabelAggregatorByItem(label_store))
pipe.add_component("Eval", ItemRecallEvaluator())

pipe.connect("Retriever.documents", "Eval.documents")
pipe.connect("LabelAgg.relevant_item_ids", "Eval.relevant_item_ids")

In [ ]:
# スコア計算
def run_eval_pipeline(pipeline, top_k):
  scores = []
  empty_rel = 0
  for _, row in dfs["test"].iterrows():
      qid = str(row["id"])
      question = safe_str(row["question"])

      out = pipeline.run(
          data={
              "Retriever": {"query": question, "top_k": top_k},
              "LabelAgg": {"question_id": qid},
          }
      )

      ev = out["Eval"]
      if ev["total_relevant"] == 0:
          empty_rel += 1
          continue
      scores.append(ev["recall"])

  return scores, empty_rel

scores, empty_rel = run_eval_pipeline(pipe, 10)

print("evaluated:", len(scores))
print("empty_relevant:", empty_rel)
print("Recall@10:", (sum(scores)/len(scores)) if scores else None)

In [ ]:
# k についてチェック
def evaluate_retriever(pipeline, topk_values = [1, 3, 5, 10, 20]):
  topk_results = {}

  for topk in topk_values:
    scores, empty_relevant = run_eval_pipeline(pipeline, topk)
    topk_results[topk] = {"recall": (sum(scores)/len(scores))}
    print(f"top_k: {topk}, empty_relevant: {empty_relevant}")

  return pd.DataFrame.from_dict(topk_results, orient="index")

es_topk_results = evaluate_retriever(pipe)

In [ ]:
# 結果描画
import matplotlib.pyplot as plt
def plot_retriever_eval(dfs, retriever_names):
  fig, ax = plt.subplots()
  for df, retriever_name in zip(dfs, retriever_names):
    df.plot(y="recall", ax=ax, label=retriever_name)
  plt.xticks(df.index)
  plt.ylabel("Top-k Recall")
  plt.xlabel("k")
  plt.show()

plot_retriever_eval([es_topk_results], ["BM25"])

In [ ]:
from haystack_integrations.components.retrievers.elasticsearch.embedding_retriever import ElasticsearchEmbeddingRetriever
# DensePassageRetriever 用意
## haystack 2.x 系では EmbeddingRetriever に集約されている
from haystack.components.embedders import SentenceTransformersDocumentEmbedder, SentenceTransformersTextEmbedder

## 埋め込みベクトルストア
embedding_dim = 768
emb_store = ElasticsearchDocumentStore(
    hosts="http://localhost:9200",
    index="emb_store",
    custom_mapping={
        "properties": {
            "embedding": {
              "type": "dense_vector",
              "index": True,
              "dims": embedding_dim,
              "similarity": "cosine",
          },
          "content": {"type": "text",},
          "id": {"type": "keyword"},
          "item_id": {"type": "keyword"},
          "question_id": {"type": "keyword"},
          "split": {"type": "keyword"}
      }
    },
)

## 埋め込み
doc_embedder = SentenceTransformersDocumentEmbedder(
  model="sentence-transformers/multi-qa-mpnet-base-dot-v1"
)
doc_embedder.warm_up()

embedded_docs = doc_embedder.run(documents=label_docs)
emb_store.write_documents(embedded_docs["documents"], policy=DuplicatePolicy.SKIP)

## クエリ埋め込み
query_embedder = SentenceTransformersTextEmbedder(
    model="sentence-transformers/multi-qa-mpnet-base-dot-v1"
)

## レトリーバー
from typing import List
@component
class ES7EmbeddingRetriever:
    def __init__(self, document_store: ElasticsearchDocumentStore, top_k: int = 3):
        self.document_store = document_store
        self.top_k = top_k

    @component.output_types(documents=List[Document])
    def run(self, query_embedding: List[float], top_k: int = 0):
        k = top_k if top_k > 0 else self.top_k
        body = {
            "size": k,
            "query": {
                "script_score": {
                    "query": {"match_all": {}},
                    "script": {
                        "source": "cosineSimilarity(params.q, 'embedding') + 1.0",
                        "params": {"q": query_embedding},
                    },
                }
            },
        }
        res = self.document_store._client.search(
            index=self.document_store._index,
            body=body,
        )

        docs = []
        for hit in res["hits"]["hits"]:
            src = hit["_source"]
            meta = {kk: vv for kk, vv in src.items() if kk not in ("content", "embedding")}
            docs.append(
                Document(
                    id=src.get("id"),
                    content=src.get("content", ""),
                    meta=meta,
                    score=hit.get("_score"),
                )
            )
        return {"documents": docs}

emb_retriever = ES7EmbeddingRetriever(
    document_store=emb_store,
    top_k=3,
)

In [ ]:
# 評価パイプライン
emb_pip = Pipeline()
emb_pip.add_component("QueryEmbedder", query_embedder)
emb_pip.add_component("Retriever", emb_retriever)
emb_pip.add_component("LabelAgg", LabelAggregatorByItem(emb_store))
emb_pip.add_component("Eval", ItemRecallEvaluator())

emb_pip.connect("QueryEmbedder.embedding", "Retriever.query_embedding")
emb_pip.connect("Retriever.documents", "Eval.documents")
emb_pip.connect("LabelAgg.relevant_item_ids", "Eval.relevant_item_ids")

In [ ]:
# 結果
def run_eval_emb_pipeline(pipeline, top_k):
  scores = []
  empty_rel = 0
  for _, row in dfs["test"].iterrows():
      qid = str(row["id"])
      question = safe_str(row["question"])

      out = pipeline.run(
          data={
              "QueryEmbedder": {"text": question},
              "Retriever": {"top_k": top_k},
              "LabelAgg": {"question_id": qid},
          }
      )

      ev = out["Eval"]
      if ev["total_relevant"] == 0:
          empty_rel += 1
          continue
      scores.append(ev["recall"])

  return scores, empty_rel

# k についてチェック
def evaluate_emb_retriever(pipeline, topk_values = [1, 3, 5, 10, 20]):
  topk_results = {}

  for topk in topk_values:
    scores, empty_relevant = run_eval_emb_pipeline(pipeline, topk)
    topk_results[topk] = {"recall": (sum(scores)/len(scores))}
    print(f"top_k: {topk}, empty_relevant: {empty_relevant}")

  return pd.DataFrame.from_dict(topk_results, orient="index")

emb_topk_results = evaluate_emb_retriever(emb_pip)
plot_retriever_eval([es_topk_results, emb_topk_results], ["BM25", "DPR"])